# DS3 · Estadística bayesiana

**Caso:** TiendaSur quiere (1) decidir si despliega una ficha de producto nueva tras un test A/B, y (2) estimar la conversión de **25 tiendas físicas**, algunas con muy pocas visitas. Con el enfoque bayesiano obtendrás respuestas que un directivo entiende: «hay un 97 % de probabilidad de que B sea mejor» y «si me equivoco, pierdo de media 0,005 puntos de conversión».

Plan: conjugadas (Beta-Binomial) → A/B con probabilidad y pérdida esperada → MCMC a mano (Metropolis) → PyMC → modelo jerárquico → diagnóstico → comparación con el enfoque frecuentista.

> **Qué usamos:** PyMC y ArviZ (se instalan con `pip install pymc arviz`). Todo muestrea en CPU en segundos. Las trazas y los histogramas los dibujamos con matplotlib para que veas qué hay dentro de los resultados de ArviZ.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (9, 3.4), "axes.grid": True, "grid.alpha": 0.3})
rng = np.random.default_rng(2026)

try:
    import arviz as az
    import pymc as pm
    print(f"PyMC {pm.__version__} · ArviZ {az.__version__}")
except ImportError:                       # p. ej. en Colab limpio
    raise SystemExit("Instala las librerías con:  %pip install pymc arviz   y reinicia el kernel")

## 1 · Beta-Binomial: la conjugada

Con una conversión desconocida `p`, un *prior* `Beta(a, b)` y `k` compras en `n` visitas, el posterior es **exactamente** `Beta(a + k, b + n − k)`. Interpretación: `a` y `b` son «compras y no compras imaginarias» previas; `a + b` es el **peso** del prior en visitas.

Comparamos tres priors con pocos datos (5/50) y con muchos (100/2000): con pocos datos el prior manda; con muchos, los datos lo anulan.

In [ ]:
def posterior_beta(a, b, aciertos, n):
    """Beta(a, b) a priori + `aciertos` de `n` ensayos  ->  Beta(a + aciertos, b + n - aciertos). Sin simular."""
    return stats.beta(a + aciertos, b + n - aciertos)

# Tres priors para la conversión de una landing nueva de TiendaSur
priors = {
    "plano Beta(1,1)": (1, 1),                      # «no sé nada»: todos los valores igual de creíbles
    "débil Beta(2,58)": (2, 58),                    # ≈ 3,3 %, pesa como 60 visitas
    "informativo Beta(30,970)": (30, 970),          # histórico: 3 % con ~1000 visitas de peso
}
x = np.linspace(0, 0.15, 600)
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=False)
for ax, (k, n) in zip(ejes, [(5, 50), (100, 2000)]):
    for nombre, (a, b) in priors.items():
        ax.plot(x, posterior_beta(a, b, k, n).pdf(x), label=nombre)
    ax.axvline(k / n, color="k", ls=":", label="frecuencia observada")
    ax.set_title(f"Posterior con {k} compras en {n} visitas"); ax.set_xlabel("conversión")
ejes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

filas = []
for (k, n) in [(5, 50), (100, 2000)]:
    for nombre, (a, b) in priors.items():
        post = posterior_beta(a, b, k, n)
        lo, hi = post.ppf([0.03, 0.97])
        filas.append({"datos": f"{k}/{n}", "prior": nombre, "media": post.mean(), "IC creíble 94 %": f"[{lo:.3f}, {hi:.3f}]"})
print(pd.DataFrame(filas).round(4).to_string(index=False))

### Frecuentista frente a bayesiano con los mismos datos
Los números casi coinciden (con prior plano), la **interpretación** no.

In [ ]:
# ¿Qué dice el enfoque frecuentista con esos mismos datos? (5 compras en 50 visitas)
k, n = 5, 50
ic_wilson = stats.binomtest(k, n).proportion_ci(confidence_level=0.94, method="wilson")
print(f"Frecuentista: estimación {k/n:.3f}, IC Wilson 94 % = [{ic_wilson.low:.3f}, {ic_wilson.high:.3f}]")
post = posterior_beta(1, 1, k, n)
print(f"Bayesiano (prior plano): media {post.mean():.3f}, intervalo creíble 94 % = {np.round(post.ppf([.03, .97]), 3)}")
print(f"\nP(conversión > 8 %) según el posterior: {1 - post.cdf(0.08):.2f}   <- una probabilidad sobre el parámetro")
print("Un IC frecuentista NO dice «hay un 94 % de probabilidad de que p esté dentro»: dice que el procedimiento acierta el 94 % de las veces.")

## 2 · A/B bayesiano: probabilidad, pérdida esperada y regla de decisión

Simulamos muchas veces `(pA, pB)` del posterior y respondemos:
- `P(B > A)`: probabilidad de que B sea mejor.
- **Pérdida esperada** de elegir B: `E[max(pA − pB, 0)]`, lo que pierdo de media (en puntos de conversión) si elijo B y resulta que A era mejor.
- **Regla de decisión** acordada *antes* del test: despliega B si su pérdida esperada es menor que un umbral ε que el negocio considera irrelevante (aquí 0,05 puntos de conversión).

In [ ]:
# Test A/B de la página de producto: A = actual, B = nueva ficha con vídeo
kA, nA = 120, 4000
kB, nB = 150, 4000
PRIOR = (2, 58)          # débil y razonable: conversión típica ~3 % (histórico de TiendaSur), peso de 60 visitas

def decision_ab(kA, nA, kB, nB, prior=PRIOR, eps=0.0005, n_sim=200_000, semilla=1):
    """Compara dos conversiones por simulación del posterior (conjugado Beta-Binomial)."""
    g = np.random.default_rng(semilla)
    pA = posterior_beta(*prior, kA, nA).rvs(n_sim, random_state=g)
    pB = posterior_beta(*prior, kB, nB).rvs(n_sim, random_state=g)
    perdida_B = np.clip(pA - pB, 0, None).mean()       # lo que pierdo (en puntos de conversión) si elijo B y A era mejor
    perdida_A = np.clip(pB - pA, 0, None).mean()       # ... si elijo A y B era mejor
    return {
        "P(B > A)": float((pB > pA).mean()),
        "uplift relativo medio": float(((pB - pA) / pA).mean()),
        "intervalo uplift 94 %": tuple(round(float(v), 3) for v in np.quantile((pB - pA) / pA, [0.03, 0.97])),
        "pérdida esperada si elijo B": float(perdida_B),
        "pérdida esperada si elijo A": float(perdida_A),
        "decisión": "desplegar B" if perdida_B < eps else ("quedarse con A" if perdida_A < eps else "seguir recogiendo datos"),
    }, pA, pB

res, pA, pB = decision_ab(kA, nA, kB, nB)
for clave, valor in res.items():
    print(f"{clave:30s} {valor if isinstance(valor, (str, tuple)) else round(valor, 5)}")

# Para comparar: el test frecuentista de dos proporciones
z = (kB / nB - kA / nA) / np.sqrt((kA + kB) / (nA + nB) * (1 - (kA + kB) / (nA + nB)) * (1 / nA + 1 / nB))
print(f"\nFrecuentista: z = {z:.2f}, p-valor bilateral = {2 * (1 - stats.norm.cdf(abs(z))):.3f}")

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.2))
ejes[0].hist(pA, 80, density=True, alpha=.6, label="A"); ejes[0].hist(pB, 80, density=True, alpha=.6, label="B")
ejes[0].set_title("Posterior de la conversión"); ejes[0].legend()
ejes[1].hist((pB - pA)[:50000] * 100, 80, color="C2"); ejes[1].axvline(0, color="k")
ejes[1].set_title("Diferencia B - A (puntos porcentuales)"); plt.tight_layout(); plt.show()

### El peligro de mirar cada día
Si miras el test a diario y paras en cuanto «sale significativo», fabricas falsos positivos. Lo comprobamos con tests A/A (sin diferencia real).

In [ ]:
# ¿Qué pasa si miras el test cada día y paras en cuanto «sale»? Simulamos 2000 tests A/A (no hay diferencia real).
g = np.random.default_rng(7)
n_tests, dias, visitas_dia, p_real = 2000, 20, 200, 0.03
cA = g.binomial(visitas_dia, p_real, (n_tests, dias)).cumsum(1)
cB = g.binomial(visitas_dia, p_real, (n_tests, dias)).cumsum(1)
nn = visitas_dia * np.arange(1, dias + 1)

# Frecuentista: z de dos proporciones en cada mirada
pool = (cA + cB) / (2 * nn)
z = (cB / nn - cA / nn) / np.sqrt(np.clip(pool * (1 - pool) * 2 / nn, 1e-12, None))
falso_frec_cada_dia = (np.abs(z) > 1.96).any(1).mean()
falso_frec_final = (np.abs(z[:, -1]) > 1.96).mean()

# Bayesiano: P(B>A) por aproximación normal de los posteriores (rápida, vale para n grandes)
def p_b_mayor(cA, cB, nn, prior=PRIOR):
    a1, b1, a2, b2 = prior[0] + cA, prior[1] + nn - cA, prior[0] + cB, prior[1] + nn - cB
    m1, m2 = a1 / (a1 + b1), a2 / (a2 + b2)
    v1, v2 = a1 * b1 / ((a1 + b1) ** 2 * (a1 + b1 + 1)), a2 * b2 / ((a2 + b2) ** 2 * (a2 + b2 + 1))
    return stats.norm.cdf((m2 - m1) / np.sqrt(v1 + v2))

pb = p_b_mayor(cA, cB, nn)
umbral = (pb > 0.975) | (pb < 0.025)       # umbral comparable al p<0,05 bilateral
falso_bayes_cada_dia, falso_bayes_final = umbral.any(1).mean(), umbral[:, -1].mean()
print(f"Falsos positivos mirando cada día : frecuentista (p<0,05) {falso_frec_cada_dia:.1%} | bayesiano (P>97,5 %) {falso_bayes_cada_dia:.1%}")
print(f"Falsos positivos mirando solo al final: frecuentista {falso_frec_final:.1%} | bayesiano {falso_bayes_final:.1%}")
print("Mirar y parar al primer «éxito» infla los falsos positivos en ambos enfoques; el bayesiano NO te exime de planificar la regla de parada.")

## 3 · MCMC desde cero: Metropolis

Cuando el posterior no tiene fórmula, lo **muestreamos**. Metropolis de paseo aleatorio: propones un salto, lo aceptas con probabilidad `min(1, post(nuevo)/post(actual))`. Lo probamos en un caso con solución exacta para ver si funciona, y vemos qué ocurre con un paso mal elegido: el diagnóstico (R-hat y ESS) lo delata. Nuestro R-hat y el de ArviZ difieren en el valor (ArviZ usa rangos normalizados), pero ambos distinguen cadenas mezcladas de las que no.

In [ ]:
def log_posterior(p, k, n, a=1, b=1):
    """log prior Beta(a,b) + log verosimilitud Binomial, salvo constantes."""
    if not 0 < p < 1:
        return -np.inf
    return (a - 1 + k) * np.log(p) + (b - 1 + n - k) * np.log(1 - p)

def metropolis(k, n, pasos, paso_sd, inicio, semilla):
    """Metropolis de paseo aleatorio: propone p' = p + N(0, paso_sd²); acepta con prob. min(1, post(p')/post(p))."""
    g = np.random.default_rng(semilla)
    cadena = np.empty(pasos)
    p, lp, aceptadas = inicio, log_posterior(inicio, k, n), 0
    for t in range(pasos):
        prop = p + g.normal(0, paso_sd)
        lp_prop = log_posterior(prop, k, n)
        if np.log(g.uniform()) < lp_prop - lp:
            p, lp, aceptadas = prop, lp_prop, aceptadas + 1
        cadena[t] = p
    return cadena, aceptadas / pasos

def rhat(cadenas):
    """R-hat dividido (split R-hat): compara varianza ENTRE cadenas con varianza DENTRO. ≈1 = mezcladas."""
    m, n = cadenas.shape
    mitades = cadenas[:, : n // 2 * 2].reshape(m * 2, n // 2)
    W = mitades.var(1, ddof=1).mean()
    B = (n // 2) * mitades.mean(1).var(ddof=1)
    return float(np.sqrt(((n // 2 - 1) / (n // 2) * W + B / (n // 2)) / W))

k, n = 150, 4000                       # datos de B del A/B
exacto = posterior_beta(1, 1, k, n)
inicios = [0.02, 0.2, 0.5, 0.9]        # cuatro cadenas que arrancan MUY dispersas, adrede
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.4), sharey=True)
resumen = []
for ax, (nombre, sd) in zip(ejes, [("paso demasiado pequeño (0,0003)", 0.0003), ("paso razonable (0,006)", 0.006)]):
    salidas = [metropolis(k, n, 3000, sd, ini, s) for s, ini in enumerate(inicios)]
    cad = np.array([c for c, _ in salidas])
    for c in cad:
        ax.plot(c)
    ax.set_title(nombre); ax.set_xlabel("iteración")
    util = cad[:, 500:]                              # descartamos el rodaje (burn-in)
    resumen.append({"configuración": nombre, "aceptación": round(float(np.mean([a for _, a in salidas])), 2),
                    "R-hat (propio)": round(rhat(util), 3), "R-hat (ArviZ)": round(float(az.rhat(util)), 3),
                    "ESS (ArviZ)": int(az.ess(util)), "media": round(float(util.mean()), 4)})
plt.tight_layout(); plt.show()
print(pd.DataFrame(resumen).to_string(index=False))
print(f"Exacto (conjugada): media {exacto.mean():.4f}, sd {exacto.std():.4f}")
print("Regla práctica: R-hat < 1,01 y ESS de cientos como mínimo. Paso pequeño = casi todo se acepta pero casi no se avanza.")


## 4 · PyMC: el mismo modelo en 6 líneas

PyMC define el modelo (prior + verosimilitud) y usa **NUTS**, un muestreador mucho más eficiente que Metropolis. Debe reproducir la solución exacta. Si no coincide con la conjugada, algo va mal en el modelo o en el muestreo.

In [ ]:
with pm.Model() as modelo_b:
    p = pm.Beta("p", 1, 1)                              # prior
    pm.Binomial("compras", n=n, p=p, observed=k)        # verosimilitud
    t0 = time.time()
    idata_b = pm.sample(1000, tune=1000, chains=4, random_seed=1, progressbar=False)
print(f"Muestreo: {time.time() - t0:.0f} s")
print(az.summary(idata_b, var_names=["p"]))
print(f"Exacto: media {exacto.mean():.4f}, sd {exacto.std():.4f}   |   divergencias: {int(idata_b.sample_stats['diverging'].values.sum())}")

traza = idata_b.posterior["p"].values                   # forma (cadenas, iteraciones)
fig, ejes = plt.subplots(1, 2, figsize=(11, 3))
for c in traza:
    ejes[0].plot(c, lw=.5)
    ejes[1].hist(c, 40, density=True, alpha=.4)
x2 = np.linspace(traza.min(), traza.max(), 200)
ejes[1].plot(x2, exacto.pdf(x2), "k", label="exacto"); ejes[1].legend()
ejes[0].set_title("Trazas de las 4 cadenas (deben parecer «ruido de césped»)"); plt.tight_layout(); plt.show()

In [ ]:
# El mismo A/B, ahora como modelo PyMC: la misma pregunta con una herramienta que escala a modelos sin conjugada
with pm.Model() as modelo_ab:
    pA_ = pm.Beta("pA", *PRIOR)
    pB_ = pm.Beta("pB", *PRIOR)
    pm.Deterministic("uplift", pB_ / pA_ - 1)
    pm.Binomial("comprasA", n=nA, p=pA_, observed=kA)
    pm.Binomial("comprasB", n=nB, p=pB_, observed=kB)
    idata_ab = pm.sample(1000, tune=1000, chains=4, random_seed=2, progressbar=False)

post = idata_ab.posterior
mcmc_pb = float((post["pB"] > post["pA"]).mean())
print(f"P(B > A): MCMC {mcmc_pb:.3f}  |  conjugada+simulación {res['P(B > A)']:.3f}")
print(f"Uplift medio: MCMC {float(post['uplift'].mean()):.3f}  |  conjugada {res['uplift relativo medio']:.3f}")
print(az.summary(idata_ab, var_names=["pA", "pB", "uplift"]))

## 5 · Modelos jerárquicos: *partial pooling*

Cada tienda tiene su conversión `θᵢ`, pero las tiendas de una misma cadena se parecen. El modelo jerárquico dice: `θᵢ` sale de una distribución común (con media `μ` y dispersión `σ` que **también se estiman**). Las tiendas con pocos datos se «encogen» hacia la media; las de muchos datos apenas.

Tres estrategias: **sin pooling** (cada tienda por su cuenta: ruidoso), **pooling total** (una tasa para todas: ignora diferencias) y **partial pooling** (el término medio estimado por los datos). Como simulamos, conocemos la tasa real y podemos medir quién acierta más.

In [ ]:
# TiendaSur tiene 25 tiendas físicas; las nuevas han recibido pocas visitas (con la «cámara de cola»)
g = np.random.default_rng(11)
N = 25
p_real = g.beta(3.2, 76.8, N)                                 # conversión VERDADERA de cada tienda (media ≈ 4 %)
visitas = np.clip(g.lognormal(np.log(300), 1.0, N), 25, 6000).astype(int)
compras = g.binomial(visitas, p_real)
tiendas = pd.DataFrame({"tienda": [f"T{i + 1:02d}" for i in range(N)], "visitas": visitas, "compras": compras,
                        "tasa_obs": compras / visitas, "tasa_real": p_real})
print(tiendas.sort_values("visitas").head(6).round(3).to_string(index=False))
print(f"\nVisitas por tienda: de {visitas.min()} a {visitas.max()} (mediana {int(np.median(visitas))})")
print(f"Tiendas con 0 compras: {(compras == 0).sum()} | tasa observada máxima: {tiendas.tasa_obs.max():.1%} (¡con {tiendas.sort_values('tasa_obs').visitas.iloc[-1]} visitas!)")

In [ ]:
def logit(p):
    return np.log(p / (1 - p))

with pm.Model() as modelo_h:
    mu = pm.Normal("mu", logit(0.04), 1.0)                     # tasa media de la cadena (en escala logit)
    sigma = pm.HalfNormal("sigma", 1.0)                        # cuánto varían las tiendas entre sí
    z = pm.Normal("z", 0, 1, shape=N)                          # versión «no centrada»: z estándar, escalada después
    theta = pm.Deterministic("theta", pm.math.invlogit(mu + sigma * z))
    pm.Binomial("compras", n=visitas, p=theta, observed=compras)
    t0 = time.time()
    idata_h = pm.sample(1000, tune=1000, chains=4, target_accept=0.9, random_seed=3, progressbar=False)
print(f"Muestreo: {time.time() - t0:.0f} s | divergencias: {int(idata_h.sample_stats['diverging'].values.sum())}")
resumen_h = az.summary(idata_h, var_names=["mu", "sigma"])
print(resumen_h)
rh, es = az.rhat(idata_h), az.ess(idata_h)
print(f"Peor R-hat de todo el modelo: {max(float(rh[v].max()) for v in rh.data_vars):.3f} | peor ESS: {min(float(es[v].min()) for v in es.data_vars):.0f}")

In [ ]:
theta_pm = idata_h.posterior["theta"].mean(("chain", "draw")).values        # estimación de cada tienda con partial pooling
sin_pooling = tiendas["tasa_obs"].values
pooling_total = np.full(N, compras.sum() / visitas.sum())
rmse = lambda est: float(np.sqrt(np.mean((est - p_real) ** 2)))
print("Error (RMSE) frente a la tasa real de cada tienda:")
print(f"  sin pooling (compras/visitas)      {rmse(sin_pooling):.4f}")
print(f"  pooling total (una tasa para todas) {rmse(pooling_total):.4f}")
print(f"  partial pooling (jerárquico)         {rmse(theta_pm):.4f}")
tam = pd.qcut(visitas, 3, labels=["pocas visitas", "medias", "muchas"])
por_tam = pd.DataFrame({"sin pooling": (sin_pooling - p_real) ** 2, "partial": (theta_pm - p_real) ** 2, "tam": tam}).groupby("tam", observed=True).mean() ** .5
print("\nRMSE por tamaño de tienda:"); print(por_tam.round(4))

orden = np.argsort(visitas)
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.scatter(np.arange(N), sin_pooling[orden], c="C3", label="sin pooling", zorder=3)
ax.scatter(np.arange(N), theta_pm[orden], c="C0", label="partial pooling", zorder=3)
ax.scatter(np.arange(N), p_real[orden], marker="x", c="k", label="tasa real", zorder=3)
ax.vlines(np.arange(N), sin_pooling[orden], theta_pm[orden], color="grey", alpha=.5)
ax.axhline(compras.sum() / visitas.sum(), color="C2", ls="--", label="media global")
ax.set_xticks(np.arange(N)); ax.set_xticklabels(visitas[orden], rotation=90, fontsize=7)
ax.set_xlabel("visitas de la tienda (ordenadas)"); ax.set_ylabel("conversión"); ax.legend(ncol=4, fontsize=8)
ax.set_title("El partial pooling encoge más a las tiendas con pocos datos"); plt.tight_layout(); plt.show()

### Comprobación predictiva
¿El modelo genera datos parecidos a los observados? Si los estadísticos observados caen en la zona extrema de los simulados, el modelo no capta algo.

In [ ]:
# Comprobación predictiva a posteriori: ¿el modelo genera datos que se parecen a los observados?
th = idata_h.posterior["theta"].values.reshape(-1, N)                     # (muestras, tiendas)
rep = np.random.default_rng(0).binomial(visitas[None, :], th)             # compras simuladas con el modelo
tasa_rep = rep / visitas[None, :]
for nombre, f in {"tasa máxima": lambda t: t.max(1), "tasa mínima": lambda t: t.min(1), "desviación típica": lambda t: t.std(1)}.items():
    obs = f(tiendas["tasa_obs"].values[None, :])[0]
    p_ppc = float((f(tasa_rep) >= obs).mean())
    print(f"{nombre:18s} observada {obs:.4f} | simulada {f(tasa_rep).mean():.4f} | P(simulada >= observada) = {p_ppc:.2f}")
print("Valores extremos (cerca de 0 o 1) avisarían de que el modelo no captura algún rasgo de los datos.")

## 6 · Diagnóstico de MCMC

Antes de fiarte de un posterior:
- **R-hat** < 1,01: las cadenas coinciden entre sí.
- **ESS** (tamaño muestral efectivo) de cientos como mínimo: cuántas muestras independientes equivalen a tus muestras.
- **Divergencias** = 0 (o casi): señal de geometría difícil.
- **Trazas** sin tendencias y como «ruido de césped».

Caso clásico que falla: efecto de una promoción en 8 tiendas con errores grandes. El modelo **centrado** produce divergencias; reparametrizar a **no centrado** lo arregla. (Con 3 divergencias residuales también en el no centrado, toca subir `target_accept`; los diagnósticos no son un sí/no sino una conversación.)

In [ ]:
# Diagnóstico con un caso clásico que FALLA: efecto de una promoción en 8 tiendas (Rubin, 1981: «ocho escuelas»)
efecto = np.array([28, 8, -3, 7, -1, 1, 18, 12.])        # uplift estimado de cada tienda (unidades/semana)
error = np.array([15, 10, 16, 11, 9, 11, 10, 18.])       # su error estándar

def ocho(centrado):
    with pm.Model():
        mu = pm.Normal("mu", 0, 5)
        tau = pm.HalfCauchy("tau", 5)
        if centrado:
            theta = pm.Normal("theta", mu, tau, shape=8)
        else:
            theta = pm.Deterministic("theta", mu + tau * pm.Normal("z", 0, 1, shape=8))
        pm.Normal("y", theta, error, observed=efecto)
        return pm.sample(1000, tune=1000, chains=4, target_accept=0.8, random_seed=4, progressbar=False)

idt = {"centrado": ocho(True), "no centrado": ocho(False)}
for nombre, d in idt.items():
    print(f"{nombre:12s} divergencias: {int(d.sample_stats['diverging'].values.sum()):4d} | ESS(tau): {float(az.ess(d)['tau']):6.0f} | R-hat(tau): {float(az.rhat(d)['tau']):.3f}")

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, (nombre, d) in zip(ejes, idt.items()):
    div = d.sample_stats["diverging"].values.ravel().astype(bool)
    lt, t0_ = np.log(d.posterior["tau"].values.ravel()), d.posterior["theta"].values[..., 0].ravel()
    ax.scatter(t0_[~div], lt[~div], s=4, alpha=.3); ax.scatter(t0_[div], lt[div], s=8, c="r", label="divergencia")
    ax.set_title(f"{nombre}"); ax.set_xlabel("theta[0]"); ax.legend()
ejes[0].set_ylabel("log(tau)"); plt.tight_layout(); plt.show()
print("Los puntos rojos se agolpan en la punta del «embudo» (tau pequeño): el muestreador no puede explorarla con el modelo centrado.")

## 7 · Frecuentista frente a bayesiano: cobertura

Un intervalo del 90 % debería contener el valor real el 90 % de las veces. Simulamos miles de experimentos con `p = 3 %` y medimos cobertura y anchura. Observa cómo un prior fuerte y *acertado* da intervalos más estrechos, pero uno fuerte y *equivocado* es peligroso con pocos datos. Wilson con n=100 también puede quedarse por debajo del 90 % por la naturaleza discreta de los datos.

In [ ]:
# Cuatro formas de construir un intervalo del 90 % para una conversión del 3 %: ¿cuántas veces contiene el valor real?
g = np.random.default_rng(5)
p0, reps = 0.03, 4000
def cobertura(n, metodo):
    k = g.binomial(n, p0, reps)
    if metodo == "Wilson":
        lo, hi = zip(*[(r.low, r.high) for r in (stats.binomtest(int(x), n).proportion_ci(0.90, "wilson") for x in k)])
    elif metodo == "Clopper-Pearson":
        lo = np.where(k == 0, 0, stats.beta.ppf(0.05, k, n - k + 1)); hi = stats.beta.ppf(0.95, k + 1, n - k)
    else:
        a, b = {"creíble prior plano": (1, 1), "creíble débil Beta(2,58)": (2, 58),
                "creíble informativo ACERTADO": (3, 97), "creíble informativo EQUIVOCADO (10 %)": (20, 180)}[metodo]
        lo, hi = stats.beta.ppf(0.05, a + k, b + n - k), stats.beta.ppf(0.95, a + k, b + n - k)
    lo, hi = np.asarray(lo), np.asarray(hi)
    return ((lo <= p0) & (p0 <= hi)).mean(), (hi - lo).mean()

metodos = ["Wilson", "Clopper-Pearson", "creíble prior plano", "creíble débil Beta(2,58)",
           "creíble informativo ACERTADO", "creíble informativo EQUIVOCADO (10 %)"]
filas = [{"n": n, "método": m, **dict(zip(["cobertura", "ancho"], np.round(cobertura(n, m), 3)))} for n in (30, 100, 500) for m in metodos]
print(pd.DataFrame(filas).pivot(index="método", columns="n", values=["cobertura", "ancho"]).loc[metodos].to_string())
print("\nObjetivo de cobertura: 0.90. Un prior equivocado y fuerte rompe la cobertura con pocos datos; con n=500 los datos lo corrigen.")

## Ejercicios

1. **Prior informativo.** Histórico de la landing: conversión media 2,8 %, intervalo del 95 % ≈ [2,0 %, 3,8 %]. Encuentra la `Beta(a, b)` que lo representa y calcula el posterior con 9 compras en 200 visitas, frente a un prior plano. ¿Cuánto pesa el prior?
2. **Decisión A/B.** Test «envío gratis»: A 300/10 000, B 330/10 000. Calcula `P(B > A)`, la pérdida esperada y la decisión con ε = 0,0005. Traduce a euros (8 € de margen por compra) por 100 000 visitas.
3. **Metropolis para Poisson.** Pedidos por hora de la tienda online (24 horas dadas), prior `Gamma(2, 0,2)`. Implementa Metropolis y compara con la conjugada Gamma.
4. **Tienda nueva.** Con el modelo jerárquico, da la distribución de la conversión de una tienda **sin datos**, y compara sin pooling y partial pooling en la tienda con menos visitas.
5. **Sensibilidad.** Cambia el prior de `σ` (HalfNormal con escala 0,1 / 1 / 5) y observa el efecto en la tienda menor y en el error global.

In [ ]:
# Ejercicio 1 · tu código aquí


In [ ]:
# Ejercicio 2 · tu código aquí


In [ ]:
# Ejercicio 3 · tu código aquí


In [ ]:
# Ejercicio 4 · tu código aquí


In [ ]:
# Ejercicio 5 · tu código aquí
